In [1]:
print("Let's Start")

Let's Start


In [2]:
from sklearn.datasets import load_wine
x ,  y = load_wine(return_X_y= True , as_frame=True)
x.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [4]:
set(y)

{0, 1, 2}

In [5]:
from sklearn.model_selection import train_test_split
x_train , x_text , y_train , y_text = train_test_split(x , y , test_size=0.2 , random_state=0 )


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
lr = LogisticRegression(solver='lbfgs', max_iter=1000)
lr.fit(x_train , y_train)
y_pred = lr.predict(x_text)
accuracy_score(y_text , y_pred)

/mnt/WindowsD/Artificial Intelligence/genai/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.9444444444444444

In [10]:
lr.coef_ , lr.intercept_


(array([[-0.43911957,  0.41017734,  0.85136334, -0.29020203, -0.01502096,
         -0.12858751,  0.97711715,  0.13760963, -0.24250112,  0.12034264,
         -0.12888689,  0.79132698,  0.01002894],
        [ 0.57441301, -0.65890628, -0.95398677,  0.20678166, -0.01045227,
          0.29462847,  0.08806526, -0.02481712,  0.88811934, -1.03890596,
          0.43155829, -0.04703468, -0.00738075],
        [-0.13529343,  0.24872894,  0.10262343,  0.08342037,  0.02547323,
         -0.16604096, -1.06518241, -0.11279252, -0.64561822,  0.91856332,
         -0.30267141, -0.74429231, -0.00264818]]),
 array([-1.20540029,  1.57979728, -0.37439699]))

In [ ]:
import numpy as np


class MyLogisticRegression:

    def __init__(self, iterations=100, lr_=0.1):
        if iterations <= 0:
            raise ValueError("iterations must be greater than 0.")

        if lr_ <= 0:
            raise ValueError("lr_ must be greater than 0.")

        self.coef_ = None
        self.intercept_ = None

        self.iterations = iterations
        self.lr_ = lr_

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, x, y):

        x = np.asarray(x, dtype=float)
        y = np.asarray(y)

        if x.ndim != 2:
            raise ValueError(
                "X must be a 2-dimensional array."
            )

        if y.ndim != 1:
            y = y.ravel()

        if x.shape[0] != y.shape[0]:
            raise ValueError(
                "X and y must have the same number of samples."
            )

        if len(np.unique(y)) != 2:
            raise ValueError(
                "MyLogisticRegression supports binary classification only."
            )

        if not np.all(np.isin(y, [0, 1])):
            raise ValueError(
                "For binary classification, y must contain only 0 and 1."
            )

        y = y.astype(float)

        n_samples, n_features = x.shape

        weights = np.zeros(n_features)
        intercept = 0.0

        for _ in range(self.iterations):

            # Linear equation
            z = np.dot(x, weights) + intercept

            # Probability
            y_hat = self._sigmoid(z)

            # Gradient
            error = y_hat - y

            dw = np.dot(x.T, error) / n_samples
            db = np.sum(error) / n_samples

            # Gradient descent
            weights -= self.lr_ * dw
            intercept -= self.lr_ * db

        print("Model Trained Successfully")
        self.coef_ = weights
        self.intercept_ = intercept

        return self.coef_ , self.intercept_

    def predict_proba(self, x):

        if self.coef_ is None:
            raise ValueError(
                "Model must be fitted before prediction."
            )

        x = np.asarray(x, dtype=float)

        if x.ndim != 2:
            raise ValueError(
                "X must be a 2-dimensional array."
            )

        if x.shape[1] != len(self.coef_):
            raise ValueError(
                f"Expected {len(self.coef_)} features, "
                f"got {x.shape[1]}."
            )

        z = np.dot(x, self.coef_) + self.intercept_

        return self._sigmoid(z)

    def predict(self, x, threshold=0.5):

        if not 0 < threshold < 1:
            raise ValueError(
                "threshold must be between 0 and 1."
            )

        probabilities = self.predict_proba(x)

        return (probabilities >= threshold).astype(int)